# Merge monthly Chuuchuu outputs

1. **Summary stats**: vertically appends every `summary_stats/chuuchuu_summary_YYYY_MM.xlsx` into a single Excel.
Each sheet (`Summary`, `SNCF`, `DB`, ...) is merged across months separately, so the output keeps the same sheets and the same columns as the originals.
2. **Enriched parquet**: vertically merges every `data_chuuchuu_YYYY_MM_enriched.parquet` of the shared-drive `processed_data/` folder (see `chuuchuu_paths.py`) into one parquet file, written to the same folder.

## 1. Summary stats (Excel)

In [2]:
import re
from pathlib import Path

import pandas as pd

SUMMARY_DIR = Path("summary_stats")
OUTPUT_FILE = SUMMARY_DIR / "chuuchuu_summary_monthly_merged.xlsx"

# Only the monthly files (excludes *_full, *_french, etc.)
pattern = re.compile(r"^chuuchuu_summary_(\d{4})_(\d{2})\.xlsx$")
monthly_files = sorted(f for f in SUMMARY_DIR.iterdir() if pattern.match(f.name))
print(f"{len(monthly_files)} monthly files found:")
for f in monthly_files:
    print(" -", f.name)

12 monthly files found:
 - chuuchuu_summary_2025_01.xlsx
 - chuuchuu_summary_2025_02.xlsx
 - chuuchuu_summary_2025_03.xlsx
 - chuuchuu_summary_2025_04.xlsx
 - chuuchuu_summary_2025_05.xlsx
 - chuuchuu_summary_2025_06.xlsx
 - chuuchuu_summary_2025_07.xlsx
 - chuuchuu_summary_2025_08.xlsx
 - chuuchuu_summary_2025_09.xlsx
 - chuuchuu_summary_2025_10.xlsx
 - chuuchuu_summary_2025_11.xlsx
 - chuuchuu_summary_2025_12.xlsx


In [3]:
# Read every sheet of every file, keeping the sheet order and column order of the first file
merged = {}
reference_columns = {}

for f in monthly_files:
    sheets = pd.read_excel(f, sheet_name=None)
    for sheet_name, df in sheets.items():
        if sheet_name not in reference_columns:
            reference_columns[sheet_name] = list(df.columns)
        elif list(df.columns) != reference_columns[sheet_name]:
            print(f"WARNING: column mismatch in {f.name} / {sheet_name}")
        merged.setdefault(sheet_name, []).append(df)

merged = {
    sheet_name: pd.concat(dfs, ignore_index=True)[reference_columns[sheet_name]]
    for sheet_name, dfs in merged.items()
}

for sheet_name, df in merged.items():
    print(f"{sheet_name:<12} {df.shape}")

Summary      (2350, 21)
SNCF         (271, 22)
DB           (783, 22)
NS           (194, 22)
OEBB         (625, 22)
SBB          (557, 22)
SNCB         (187, 22)
Trenitalia   (394, 22)
Summary_partial_alt (2350, 23)


In [4]:
# Sanity check: merged row count = sum of monthly row counts, all months present
for sheet_name, df in merged.items():
    expected = sum(len(pd.read_excel(f, sheet_name=sheet_name)) for f in monthly_files)
    assert len(df) == expected, f"{sheet_name}: {len(df)} rows vs {expected} expected"

merged["Summary"].groupby(["Year", "Month"]).size()

Year  Month
2025  1        219
      2        184
      3        181
      4        198
      5        192
      6        183
      7        182
      8        181
      9        188
      10       203
      11       207
      12       232
dtype: int64

In [5]:
with pd.ExcelWriter(OUTPUT_FILE) as writer:
    for sheet_name, df in merged.items():
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Saved to {OUTPUT_FILE}")

Saved to summary_stats\chuuchuu_summary_monthly_merged.xlsx


## 2. Enriched parquet

Streams row group by row group (the full year is ~80M rows), so memory stays low.
Schemas are unified first: e.g. `routeNumber_int` is int64 in some months and double in others (months containing NaNs) -> cast to the common type.

In [6]:
import pyarrow as pa
import pyarrow.parquet as pq

from chuuchuu_paths import ENRICHED_DIR, output_path

PARQUET_DIR = ENRICHED_DIR
# Only monthly files (data_chuuchuu_YYYY_MM_enriched.parquet), so a previous merged output is never re-ingested
parquet_pattern = re.compile(r"^data_chuuchuu_(\d{4}_\d{2})_enriched\.parquet$")
DROP_COLUMNS = ["__index_level_0__"]  # per-file pandas index, meaningless once stacked

parquet_files = sorted(f for f in PARQUET_DIR.iterdir() if parquet_pattern.match(f.name))
if not parquet_files:
    raise FileNotFoundError(f"No monthly parquet files found in {PARQUET_DIR}")
print(f"{len(parquet_files)} monthly parquet files found:")
for f in parquet_files:
    print(" -", f.name)

first = parquet_pattern.match(parquet_files[0].name).group(1)
last = parquet_pattern.match(parquet_files[-1].name).group(1)
PARQUET_OUTPUT_FILE = output_path(f"{first}_to_{last}")


12 monthly parquet files found:
 - data_chuuchuu_2025_01_enriched.parquet
 - data_chuuchuu_2025_02_enriched.parquet
 - data_chuuchuu_2025_03_enriched.parquet
 - data_chuuchuu_2025_04_enriched.parquet
 - data_chuuchuu_2025_05_enriched.parquet
 - data_chuuchuu_2025_06_enriched.parquet
 - data_chuuchuu_2025_07_enriched.parquet
 - data_chuuchuu_2025_08_enriched.parquet
 - data_chuuchuu_2025_09_enriched.parquet
 - data_chuuchuu_2025_10_enriched.parquet
 - data_chuuchuu_2025_11_enriched.parquet
 - data_chuuchuu_2025_12_enriched.parquet


In [7]:
# Unified schema: same columns everywhere, int/float mixes promoted to float64, anything else fails loudly
schemas = [pq.read_schema(f).remove_metadata() for f in parquet_files]
names = schemas[0].names
for f, s in zip(parquet_files, schemas):
    if set(s.names) != set(names):
        raise ValueError(
            f"{f.name}: column mismatch\n  missing: {set(names) - set(s.names)}\n  extra: {set(s.names) - set(names)}"
        )

fields = []
for name in names:
    if name in DROP_COLUMNS:
        continue
    types = {s.field(name).type for s in schemas}
    if len(types) == 1:
        fields.append(schemas[0].field(name))
    elif all(pa.types.is_integer(t) or pa.types.is_floating(t) for t in types):
        fields.append(pa.field(name, pa.float64()))
        print(f"  {name}: {sorted(map(str, types))} -> double")
    else:
        raise TypeError(f"Incompatible types for column {name}: {types}")
parquet_schema = pa.schema(fields)


  routeNumber_int: ['double', 'int64'] -> double


In [8]:
# Stream row groups into a temp file, only expose the final file once fully written
tmp_output = PARQUET_OUTPUT_FILE.with_suffix(".parquet.tmp")

total_rows = 0
with pq.ParquetWriter(tmp_output, parquet_schema, compression="snappy") as writer:
    for f in parquet_files:
        pf = pq.ParquetFile(f)
        for i in range(pf.num_row_groups):
            table = pf.read_row_group(i, columns=parquet_schema.names)
            writer.write_table(table.cast(parquet_schema))
        total_rows += pf.metadata.num_rows
        print(f"  {f.name}: {pf.metadata.num_rows:,} rows (running total {total_rows:,})")

tmp_output.replace(PARQUET_OUTPUT_FILE)

# Sanity check: merged row count = sum of monthly row counts
merged_rows = pq.ParquetFile(PARQUET_OUTPUT_FILE).metadata.num_rows
assert merged_rows == total_rows, f"Row count mismatch: {merged_rows:,} vs {total_rows:,}"
print(f"\nWrote {merged_rows:,} rows, {len(parquet_schema)} columns -> {PARQUET_OUTPUT_FILE}")


  data_chuuchuu_2025_01_enriched.parquet: 6,525,878 rows (running total 6,525,878)
  data_chuuchuu_2025_02_enriched.parquet: 5,760,232 rows (running total 12,286,110)
  data_chuuchuu_2025_03_enriched.parquet: 6,327,517 rows (running total 18,613,627)
  data_chuuchuu_2025_04_enriched.parquet: 5,871,692 rows (running total 24,485,319)
  data_chuuchuu_2025_05_enriched.parquet: 6,660,113 rows (running total 31,145,432)
  data_chuuchuu_2025_06_enriched.parquet: 6,279,107 rows (running total 37,424,539)
  data_chuuchuu_2025_07_enriched.parquet: 6,455,222 rows (running total 43,879,761)
  data_chuuchuu_2025_08_enriched.parquet: 6,095,679 rows (running total 49,975,440)
  data_chuuchuu_2025_09_enriched.parquet: 6,716,338 rows (running total 56,691,778)
  data_chuuchuu_2025_10_enriched.parquet: 8,317,387 rows (running total 65,009,165)
  data_chuuchuu_2025_11_enriched.parquet: 8,120,910 rows (running total 73,130,075)
  data_chuuchuu_2025_12_enriched.parquet: 8,845,314 rows (running total 81,97